# Week 52, Day 2: Interval problems

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w052d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
talks = [(9, 12), (10, 11), (11, 13), (12, 14), (13, 15)]

def choose(talks, key):
    chosen, free_from = [], float("-inf")
    for s, e in sorted(talks, key=key):
        if s >= free_from:
            chosen.append((s, e))
            free_from = e
    return chosen

print(choose(talks, key=lambda t: t[0]))         # earliest start
print(choose(talks, key=lambda t: t[1] - t[0]))  # shortest
print(choose(talks, key=lambda t: t[1]))         # earliest finish

In [ ]:
def most_talks(talks):
    chosen, free_from = [], float("-inf")
    for s, e in sorted(talks, key=lambda t: t[1]):
        if s >= free_from:
            chosen.append((s, e))
            free_from = e
    return chosen

festival = [(1, 4), (3, 5), (0, 6), (5, 7), (3, 9), (5, 9), (6, 10), (8, 11), (8, 12), (2, 14), (12, 16)]
print(most_talks(festival))

In [ ]:
def stab_points(intervals):
    points = []
    for s, e in sorted(intervals, key=lambda t: t[1]):
        if not points or s > points[-1]:          # not stabbed by the last point
            points.append(e)
    return points

balloons = [(10, 16), (2, 8), (1, 6), (7, 12)]
print(stab_points(balloons))

In [ ]:
def busiest(intervals):
    events = sorted([(s, +1) for s, e in intervals] + [(e, -1) for s, e in intervals])
    count = best = 0
    when = None
    for t, change in events:
        count += change
        if count > best:
            best, when = count, t
    return best, when

calls = [(9, 10), (9.5, 11), (10, 12), (10.5, 11), (12, 13)]
print(busiest(calls))

In [ ]:
def fewest_to_cover(intervals, lo, hi):
    intervals = sorted(intervals)
    chosen, covered, i = [], lo, 0
    while covered < hi:
        best = None
        while i < len(intervals) and intervals[i][0] <= covered:
            if best is None or intervals[i][1] > best[1]:
                best = intervals[i]
            i += 1
        if best is None or best[1] <= covered:
            return None                       # a gap nothing covers
        chosen.append(best)
        covered = best[1]
    return chosen

clips = [(0, 2), (4, 6), (8, 10), (1, 9), (1, 5), (5, 9)]
print(fewest_to_cover(clips, 0, 10), fewest_to_cover(clips, 0, 11))

In [ ]:
def both_free(a, b):
    i = j = 0
    out = []
    while i < len(a) and j < len(b):
        start, end = max(a[i][0], b[j][0]), min(a[i][1], b[j][1])
        if start < end:
            out.append((start, end))
        if a[i][1] < b[j][1]:
            i += 1
        else:
            j += 1
    return out

ada = [(9, 11), (13, 15), (16, 18)]
bo = [(10, 14), (17, 19)]
print(both_free(ada, bo))

> **Key idea.** Most interval problems are a sort plus a greedy sweep. For the most non-overlapping intervals, sort by finish time and take each one that starts after the last chosen finishes (an exchange argument proves it); fewest stabbing points and fewest removals follow from the same sort. For the maximum overlap, sweep +1 and −1 events in time order, breaking ties so that ends come before starts for half-open intervals. To cover a range, repeatedly take the interval reaching furthest among those starting in time. To intersect two sorted lists, use two pointers and advance whichever interval ends first. Always check whether touching intervals count as overlapping.

### Exercise: Most talks

Write `most_talks(talks)`: the largest set of non-overlapping half-open intervals, chosen by earliest finish (ties by start), as a list in time order. Then `fewest_removals(talks)`: how many must be removed so the rest don't overlap.

In [ ]:
def most_talks(talks):
    ___

def fewest_removals(talks):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef test_select():\n    \"selection\"\n    festival = [(1, 4), (3, 5), (0, 6), (5, 7), (3, 9), (5, 9), (6, 10), (8, 11), (8, 12), (2, 14), (12, 16)]\n    assert most_talks(festival) == [(1, 4), (5, 7), (8, 11), (12, 16)]\n    assert most_talks([]) == [] and fewest_removals([(1, 2), (2, 3), (1, 3)]) == 1\n    rng = random.Random(1)\n    for _ in range(100):\n        ts = [(s, s + rng.randint(1, 5)) for s in (rng.randint(0, 15) for _ in range(rng.randint(0, 8)))]\n        best = 0\n        for r in range(len(ts) + 1):\n            for combo in itertools.combinations(ts, r):\n                c = sorted(combo)\n                if all(c[k][1] <= c[k + 1][0] for k in range(len(c) - 1)):\n                    best = max(best, r)\n        sel = most_talks(ts)\n        assert len(sel) == best and all(sel[k][1] <= sel[k + 1][0] for k in range(len(sel) - 1))\n        assert fewest_removals(ts) == len(ts) - best")

### Exercise: Fewest checks

Jobs run over **closed** intervals [start, end]. A supervisor checking at moment t sees every job running then (including those starting or ending at t). Write `check_times(jobs)`: the fewest moments that see every job at least once (greedy: place a check at the end of each job not yet seen, in order of end), sorted. Then `jobs_seen(jobs, times)`: for each check time, how many jobs it sees.

In [ ]:
def check_times(jobs):
    ___

def jobs_seen(jobs, times):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef test_stab():\n    \"stabbing\"\n    balloons = [(10, 16), (2, 8), (1, 6), (7, 12)]\n    assert check_times(balloons) == [6, 12] and jobs_seen(balloons, [6, 12]) == [2, 2]\n    assert check_times([]) == [] and check_times([(1, 2), (2, 3)]) == [2]\n    rng = random.Random(2)\n    for _ in range(80):\n        js = [(s, s + rng.randint(0, 4)) for s in (rng.randint(0, 10) for _ in range(rng.randint(1, 6)))]\n        pts = check_times(js)\n        assert all(any(s <= p <= e for p in pts) for s, e in js)\n        cands = sorted({e for _, e in js})\n        best = next(r for r in range(1, len(js) + 1) for combo in itertools.combinations(cands, r)\n                    if all(any(s <= p <= e for p in combo) for s, e in js))\n        assert len(pts) == best")

### Exercise: The busiest moment

Write `busiest(intervals)` for half-open intervals: `(most, start_of_busiest, end_of_busiest)`, the largest number overlapping at once and the **first** half-open stretch during which that many overlap, with a sweep line (ends before starts at equal times). `(0, None, None)` for no intervals. Then `occupancy(intervals)`: a list of `(time, count)` pairs, the count from each event time onwards, merging events at the same time into one entry, in time order.

In [ ]:
def busiest(intervals):
    ___

def occupancy(intervals):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_sweep():\n    \"sweeping\"\n    calls = [(9, 10), (9.5, 11), (10, 12), (10.5, 11), (12, 13)]\n    assert busiest(calls) == (3, 10.5, 11)\n    assert busiest([(1, 2), (2, 3)]) == (1, 1, 2) and busiest([]) == (0, None, None)\n    assert occupancy([(1, 3), (2, 4), (3, 5)]) == [(1, 1), (2, 2), (3, 2), (4, 1), (5, 0)]\n    rng = random.Random(3)\n    for _ in range(100):\n        iv = [(s, s + rng.randint(1, 5)) for s in (rng.randint(0, 10) for _ in range(rng.randint(1, 8)))]\n        most, a, b = busiest(iv)\n        counts = {t: sum(1 for s, e in iv if s <= t < e) for t in range(0, 17)}\n        assert most == max(counts.values())\n        assert all(sum(1 for s, e in iv if s <= t < e) == most for t in range(a, b))")

### Exercise: Covering the match

Video clips are `(start, end)` stretches of a match. Write `fewest_clips(clips, length)`: the fewest clips covering the whole of [0, length], greedily by reach, as a list in the order chosen, or `None` if there's a gap. Then `uncovered(clips, length)`: the gaps in [0, length] that no clip covers, as a sorted list of `(start, end)`.

In [ ]:
def fewest_clips(clips, length):
    ___

def uncovered(clips, length):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef test_cover():\n    \"covering\"\n    clips = [(0, 2), (4, 6), (8, 10), (1, 9), (1, 5), (5, 9)]\n    assert fewest_clips(clips, 10) == [(0, 2), (1, 9), (8, 10)] and fewest_clips(clips, 11) is None\n    assert uncovered(clips, 11) == [(10, 11)] and uncovered([(2, 4), (3, 5)], 7) == [(0, 2), (5, 7)]\n    rng = random.Random(4)\n    for _ in range(100):\n        cs = [(s, s + rng.randint(1, 5)) for s in (rng.randint(0, 9) for _ in range(rng.randint(1, 7)))]\n        L = rng.randint(1, 12)\n        best = None\n        for r in range(1, len(cs) + 1):\n            for combo in itertools.combinations(cs, r):\n                if all(any(s <= x < e for s, e in combo) for x in range(L)):\n                    best = r\n                    break\n            if best:\n                break\n        got = fewest_clips(cs, L)\n        assert (got is None) == (best is None) and (got is None or len(got) == best)")

### Exercise: When are we all free?

Write `both_free(a, b)` for two sorted lists of non-overlapping half-open intervals: their intersection, with two pointers. Then `all_free(schedules, min_length)`: the intervals when **everyone** is free, by intersecting the lists one after another, keeping only stretches at least `min_length` long at the end.

In [ ]:
def both_free(a, b):
    ___

def all_free(schedules, min_length):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_free():\n    \"free time\"\n    ada = [(9, 11), (13, 15), (16, 18)]\n    bo = [(10, 14), (17, 19)]\n    assert both_free(ada, bo) == [(10, 11), (13, 14), (17, 18)]\n    assert all_free([ada, bo, [(10, 18)]], 1) == [(10, 11), (13, 14), (17, 18)] and all_free([ada, bo], 2) == []\n    assert both_free([], ada) == [] and all_free([ada], 0) == ada\n    rng = random.Random(5)\n    for _ in range(100):\n        def sched():\n            pts = sorted(rng.sample(range(30), 2 * rng.randint(0, 4)))\n            return [(pts[i], pts[i + 1]) for i in range(0, len(pts), 2)]\n        a, b = sched(), sched()\n        free = [t for t in range(30) if any(s <= t < e for s, e in a) and any(s <= t < e for s, e in b)]\n        got = both_free(a, b)\n        assert [t for s, e in got for t in range(s, e)] == free")

### Bug hunt: Earliest start isn't best

This picks talks in order of **start** time, so a long early talk blocks several short ones: this prints `2`. Sort by finish time so it prints `3`.

In [ ]:
talks = [(9, 15), (10, 11), (11, 13), (13, 15)]
chosen, free_from = [], 0
for s, e in sorted(talks, key=lambda t: t[0]):
    if s >= free_from:
        chosen.append((s, e))
        free_from = e
print(len(chosen))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"earliest finish\"\n    expect_output(\"3\")")

### Bug hunt: Starts before ends

Bookings are half-open, so one ending at 10 and another starting at 10 don't clash. This sweep puts starts **before** ends at equal times (it sorts by `-change`), counting them as overlapping: this prints `2`. Let ends come first so it prints `1`.

In [ ]:
bookings = [(8, 10), (10, 12), (12, 14)]
events = sorted([(s, +1) for s, e in bookings] + [(e, -1) for s, e in bookings], key=lambda ev: (ev[0], -ev[1]))
count = best = 0
for t, change in events:
    count += change
    best = max(best, count)
print(best)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"ends first\"\n    expect_output(\"1\")")

### Bug hunt: Covering past a gap

This covering loop takes the furthest-reaching clip among those starting in time, but doesn't notice when the best one doesn't reach any further, so it loops for ever on a gap. Here it has a guard after 10 rounds and prints `stuck`. Return `None` when no clip extends the coverage, so it prints `None`.

In [ ]:
clips = sorted([(0, 3), (2, 4), (6, 9)])
covered, chosen, rounds = 0, [], 0
while covered < 9 and rounds < 10:
    rounds += 1
    reach = max((e for s, e in clips if s <= covered), default=covered)
    chosen.append(reach)
    covered = reach
print("stuck" if rounds == 10 else chosen)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"notice the gap\"\n    expect_output(\"None\")")

### Puzzle: Activity selection

Arrange the lines into the earliest-finish-first selection. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
iv = [(1, 3), (2, 5), (4, 6), (6, 8), (5, 9)]   # half-open

p = ___   # the most non-overlapping intervals that can be chosen
q = ___   # the most overlapping at any moment
r = ___   # the intersection of [(1, 5)] and [(3, 7), (8, 9)]
s = ___   # the fewest intervals from iv covering [1, 9], or None
t = ___   # True or False: (2, 4) and (4, 6) overlap, as half-open intervals

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 3 and q == 2 and r == [(3, 5)], \"p, q and r\"\n    assert s == 3 and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: when intervals have weights

Now each booking `(start, end, value)` pays a different amount, and you want the most **total value** from non-overlapping (half-open) bookings. Earliest finish fails: one valuable long booking can beat three cheap short ones. The fix is a first taste of dynamic programming:

- Sort by end time. Let `best[i]` be the most value using only the first i bookings.
- For booking i (0-based, ending at `ends[i]`), either skip it (`best[i]`) or take it, adding its value to the best using bookings that end by its start: find how many bookings end at or before `start` with `bisect_right(ends, start)` (Week 18).
- `best[i + 1] = max(best[i], value + best[bisect_right(ends, start, 0, i)])`.

That's O(n log n). Write `best_value(bookings)` and `best_set(bookings)`: the chosen bookings in time order, rebuilt by walking back through the table. Then `greedy_value(bookings)`: what earliest-finish-first gets, for comparison. The checks compare with brute force and run 50,000 bookings.

In [ ]:
from bisect import bisect_right

def best_value(bookings):
    ___

def best_set(bookings):
    ___

def greedy_value(bookings):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random, time\ndef brute(bs):\n    best = 0\n    for r in range(len(bs) + 1):\n        for combo in itertools.combinations(bs, r):\n            c = sorted(combo)\n            if all(c[k][1] <= c[k + 1][0] for k in range(len(c) - 1)):\n                best = max(best, sum(v for _, _, v in combo))\n    return best\ndef test_weighted():\n    \"weighted\"\n    bookings = [(1, 4, 5), (3, 5, 1), (0, 6, 8), (4, 7, 4), (3, 9, 6), (5, 9, 3), (6, 10, 2), (8, 11, 4)]\n    assert best_value(bookings) == 13 and sum(v for _, _, v in best_set(bookings)) == 13\n    big_one = [(0, 10, 50), (1, 2, 1), (3, 4, 1), (5, 6, 1)]\n    assert greedy_value(big_one) == 3 and best_value(big_one) == 50\n    assert best_value([]) == 0 and best_set([]) == []\n    rng = random.Random(6)\n    for _ in range(100):\n        bs = [(s, s + rng.randint(1, 5), rng.randint(1, 10)) for s in (rng.randint(0, 12) for _ in range(rng.randint(0, 7)))]\n        assert best_value(bs) == brute(bs)\n        chosen = sorted(best_set(bs))\n        assert sum(v for _, _, v in chosen) == brute(bs) and all(chosen[k][1] <= chosen[k + 1][0] for k in range(len(chosen) - 1))\ndef test_big():\n    \"fast\"\n    rng = random.Random(7)\n    bs = [(s, s + rng.randint(1, 100), rng.randint(1, 1000)) for s in (rng.randint(0, 10**6) for _ in range(50_000))]\n    t0 = time.perf_counter()\n    best_value(bs)\n    assert time.perf_counter() - t0 < 4")

## Warm-up

Three plausible rules, three different answers. Earliest start picks the long 9 to 12 talk and loses out. Shortest happens to tie here, but it can fail too. Earliest finish gets the most talks, and it always will.

## Earliest finish first

**Activity selection**: given intervals (talks, bookings), choose as many as possible with no two overlapping. Greedy: sort by **finish time**, and take each interval that starts no earlier than the last chosen one finished.

Why finish time? Exchange argument (yesterday): take any optimal selection, and look at its first interval. The interval that finishes **earliest** of all finishes no later, so swapping it in can't create an overlap with the rest. Now the optimal selection begins with the greedy choice; repeat on what's left. So greedy chooses as many as any optimal solution: it leaves the most room for everything after.

The intervals here are **half-open**, [start, end): one talk ending at 11 and another starting at 11 don't clash. If the problem treats touching intervals as overlapping, change `>=` to `>`. Always check which the problem means.

Cost: O(n log n) for the sort, then one pass.

## Stabbing and removing

Two problems turn out to be the same as activity selection in disguise:

- **Fewest points to stab every interval** (fewest moments at which to check on a set of jobs, fewest arrows to burst a row of balloons): sort by finish; whenever an interval isn't stabbed yet, put a point at its **end**, which stabs as many later intervals as possible. The number of points equals the most pairwise non-overlapping intervals (here with closed intervals, so a point at an end touches intervals starting there).
- **Fewest intervals to remove** so the rest don't overlap: n minus the activity-selection answer.

## The sweep line

"What's the most intervals overlapping at any moment?" (the most people in a building, the most simultaneous calls, the rooms needed). Week 49 did it with a heap of end times. A **sweep line** is simpler: turn each interval into two **events**, +1 at its start and −1 at its end, sort all the events by time, and walk through them keeping a running count. The largest count is the answer.

The subtle part is **ties**. With half-open intervals, an interval ending at 10 and one starting at 10 don't overlap, so at equal times the **end** (−1) must come first. Sorting `(time, change)` tuples does that automatically, because −1 sorts before +1. For closed intervals, put starts first instead.

The sweep line is a general technique: whenever a question is about what's happening at every moment, process changes in time order instead of looking at every moment.

## Covering a range

**Fewest intervals to cover [lo, hi]** (fewest camera clips to cover a whole match, fewest shifts to staff a day): greedy by reach. Sort by start. Starting from `covered = lo`, look at all intervals starting at or before `covered`, and take the one that reaches **furthest**; move `covered` to its end. If no interval starts in time, there's a gap and covering is impossible. It's the same idea as yesterday's jump game, and the exchange argument is the same: reaching further never hurts.

## Two schedules

Two people each have a sorted list of free intervals, with no overlaps within a list. When are **both** free? Walk the two lists with two pointers (Week 41): the overlap of the current pair is `(max(starts), min(ends))` if that's non-empty; then advance whichever interval ends first, since it can't overlap anything further in the other list. O(m + n).

## Practice

## Quiz

## Daily challenge

## Before you go

Look at a week of your own calendar. What's the busiest moment, and how many of your appointments could you have attended if they'd all been optional? Tomorrow: backtracking, going much further than Week 16, with a general recipe for searching through choices.